In [63]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# Create a Spark session
spark = SparkSession.builder.appName("YourAnalysis").getOrCreate()

# Load your dataset
df = spark.read.csv("DataCoSupplyChainDataset.csv", header=True, inferSchema=True)

In [17]:
# Alternatively, you can use os.path.join to create the file path
import os
directory ="content"
filename ="DataCoSupplyChainDataset.csv"
file_path = os.path.join(directory, filename)
file_path = "/content/DataCoSupplyChainDataset.csv"
file_path = "/content/DataCoSupplyChainDataset.csv"
# Import necessary libraries
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("DataCoSupplyChainDataset Analysis ") \
    .getOrCreate()

# Load the dataset into a Spark DataFrame
df = spark.read.csv("DataCoSupplyChainDataset.csv", header=True, inferSchema=True)

# Display the first few rows of the DataFrame
df.show()

+--------+------------------------+-----------------------------+-----------------+------------------+-----------------+------------------+-----------+--------------+--------------+----------------+--------------+--------------+-----------+--------------+-----------------+----------------+--------------+--------------------+----------------+-------------+---------------+-----------+------------+------------+----------+-------------+-----------------+-----------------------+--------+----------------------+-------------------+------------------------+-------------+------------------------+-----------------------+-------------------+------+----------------+----------------------+--------------+--------------------+---------------+-------------+---------------+-------------------+-------------------+--------------------+------------+-------------+--------------+--------------------------+--------------+
|    Type|Days for shipping (real)|Days for shipment (scheduled)|Benefit per order|Sale

In [5]:
!pip install pyspark



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=82bfb25e466fbcc9d16ae67190dc03d3dd306684ff3659fcd19f576ca35e0311
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [ ]:
!pip install findspark

In [38]:
#check for null values in the dataset
data_frame = df.toPandas()

for column in data_frame.columns:
    print(column, data_frame[column].isnull().sum())

Type 0
Days for shipping (real) 0
Days for shipment (scheduled) 0
Benefit per order 0
Sales per customer 0
Delivery Status 0
Late_delivery_risk 0
Category Id 0
Category Name 0
Customer City 0
Customer Country 0
Customer Email 0
Customer Fname 0
Customer Id 0
Customer Lname 8
Customer Password 0
Customer Segment 0
Customer State 0
Customer Street 0
Customer Zipcode 3
Department Id 0
Department Name 0
Latitude 0
Longitude 0
Market 0
Order City 0
Order Country 0
Order Customer Id 0
order date (DateOrders) 0
Order Id 0
Order Item Cardprod Id 0
Order Item Discount 0
Order Item Discount Rate 0
Order Item Id 0
Order Item Product Price 0
Order Item Profit Ratio 0
Order Item Quantity 0
Sales 0
Order Item Total 0
Order Profit Per Order 0
Order Region 0
Order State 0
Order Status 0
Order Zipcode 155679
Product Card Id 0
Product Category Id 0
Product Description 180519
Product Image 0
Product Name 0
Product Price 0
Product Status 0
shipping date (DateOrders) 0
Shipping Mode 0
my_features 0


In [64]:
# Feature Engineering - Assuming you want to use 'Days for shipping' and 'Sales per customer' as features
feature_cols = ['Days for shipping (real)', 'Sales per customer']
assembler = VectorAssembler(inputCols=feature_cols, outputCol='my_features')  # Change 'features' to 'my_features'
df = assembler.transform(df)

final_data = df.select('Days for shipping (real)', 'Sales per customer', 'Benefit per order')

In [65]:
# Split the data into training and testing sets
train_data, test_data = final_data.randomSplit([0.8, 0.2])

In [66]:
# Create a Linear Regression model
lr = LinearRegression(featuresCol='my_features', labelCol='Benefit per order')  # Change 'features' to 'my_features'

In [67]:
# Create a pipeline
pipeline = Pipeline(stages=[assembler, lr])

In [68]:
# Train the model
model = pipeline.fit(train_data)

In [69]:
#Linear Regression
# Evaluate the model
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(labelCol='Benefit per order', metricName='rmse')
predictions=model.transform(test_data)
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")

Root Mean Squared Error (RMSE): 101.3405225136131


In [71]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator



# Split the data into training and testing sets
train_data, test_data = final_data.randomSplit([0.8, 0.2])
# Train the model
model = pipeline.fit(train_data)
# Create a pipeline
pipeline = Pipeline(stages=[assembler, lr])



# make the predictions
predictions = model.transform(test_data)


# Evaluate the model
evaluator = MulticlassClassificationEvaluator(labelCol="Benefit per order", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print("Dataco supplychain dataset")
print("Accuracy: {:.2f}".format(accuracy))


Dataco supplychain dataset
Accuracy: 0.00


In [72]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator




# Split the data into training and testing sets
train_data, test_data = final_data.randomSplit([0.8, 0.2])
# Train the model
model = pipeline.fit(train_data)
# Create a pipeline
pipeline = Pipeline(stages=[assembler, lr])

# make the predictions
predictions = model.transform(test_data)
# Define the Linear Regression model
lr = LinearRegression(labelCol='Benefit per order')


# Evaluate the model
lr_evaluator = RegressionEvaluator(labelCol='Benefit per order', predictionCol='prediction', metricName='rmse')
rmse_lr = lr_evaluator.evaluate(predictions)
print("Root Mean Squared Error (Linear Regression) = ", rmse_lr)

Root Mean Squared Error (Linear Regression) =  102.00397706017206


In [76]:
from pyspark.ml.evaluation import RegressionEvaluator

# Assuming you have already trained and evaluated various models, storing their RMSE values
# Example RMSE values for each model
rmse_values = {
    'Linear Regression': rmse_lr,
    'Random Forest': rmse_rf

    # Add more models and their RMSE values here
}

# Print RMSE values for each model
for model, rmse in rmse_values.items():
    print(f"RMSE for {model}: {rmse}")

RMSE for Linear Regression: 102.00397706017206
RMSE for Random Forest: 102.2848121383135


In [74]:
from pyspark.ml.regression import RandomForestRegressor


# Split the data into training and testing sets
train_data, test_data = final_data.randomSplit([0.8, 0.2])
# Train the model
model = pipeline.fit(train_data)
# Create a pipeline
pipeline = Pipeline(stages=[assembler, lr])

# make the predictions
predictions = model.transform(test_data)

# Evaluate the model
rf_evaluator = RegressionEvaluator(labelCol="Benefit per order", predictionCol="prediction", metricName="rmse")
rmse_rf = rf_evaluator.evaluate(predictions)
print("Root Mean Squared Error (Random Forest Regression) = ", rmse_rf)

Root Mean Squared Error (Random Forest Regression) =  102.2848121383135


In [77]:
df.printSchema();

root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = true)
 |-- Department Id: integer (nullable = 

In [ ]:
# Stop the Spark session
spark.stop()